In [11]:
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel

In [12]:
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel

In [13]:
df = pd.read_csv('npr.csv')
df.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [14]:
print(df.shape)
print(df.columns)

(11992, 1)
Index(['Article'], dtype='object')


In [15]:
print(df.shape)
documents = df['Article'].dropna().astype(str).tolist()
print("Number of documents:", len(documents))
print(documents[0][:1000])

(11992, 1)
Number of documents: 11992
In the Washington of 2016, even when the policy can be bipartisan, the politics cannot. And in that sense, this year shows little sign of ending on Dec. 31. When President Obama moved to sanction Russia over its alleged interference in the U. S. election just concluded, some Republicans who had long called for similar or more severe measures could scarcely bring themselves to approve. House Speaker Paul Ryan called the Obama measures ”appropriate” but also ”overdue” and ”a prime example of this administration’s ineffective foreign policy that has left America weaker in the eyes of the world.” Other GOP leaders sounded much the same theme. ”[We have] been urging President Obama for years to take strong action to deter Russia’s worldwide aggression, including its   operations,” wrote Rep. Devin Nunes,  . chairman of the House Intelligence Committee. ”Now with just a few weeks left in office, the president has suddenly decided that some stronger measu

In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

print(preprocessed_documents[0][:50])

['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker']


In [17]:
dictionary = corpora.Dictionary(preprocessed_documents)
dictionary.filter_extremes(no_below=15, no_above=0.5)

corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print("Vocabulary size:", len(dictionary))
print("First BOW document:", corpus[0][:20])

Vocabulary size: 15974
First BOW document: [(0, 1), (1, 3), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 4), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 3), (16, 1), (17, 1), (18, 2), (19, 1)]


In [18]:
lda_model = LdaModel(
    corpus=corpus,
    num_topics=5,
    id2word=dictionary,
    passes=15,
    random_state=42
)

In [19]:
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

Top terms for Topic #0:
['police', 'report', 'country', 'government', 'state', 'attack', 'told', 'official', 'city', 'officer']

Top terms for Topic #1:
['food', 'study', 'water', 'disease', 'health', 'patient', 'university', 'human', 'research', 'may']

Top terms for Topic #2:
['state', 'school', 'health', 'company', 'law', 'student', 'percent', 'care', 'program', 'federal']

Top terms for Topic #3:
['trump', 'clinton', 'president', 'republican', 'campaign', 'state', 'election', 'vote', 'obama', 'voter']

Top terms for Topic #4:
['know', 'think', 'thing', 'life', 'woman', 'really', 'story', 'show', 'day', 'back']



In [20]:
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "police" (weight: 0.007)
- "report" (weight: 0.006)
- "country" (weight: 0.005)
- "government" (weight: 0.005)
- "state" (weight: 0.004)
- "attack" (weight: 0.004)
- "told" (weight: 0.004)
- "official" (weight: 0.004)
- "city" (weight: 0.004)
- "officer" (weight: 0.003)

Topic 1:
- "food" (weight: 0.006)
- "study" (weight: 0.006)
- "water" (weight: 0.004)
- "disease" (weight: 0.004)
- "health" (weight: 0.003)
- "patient" (weight: 0.003)
- "university" (weight: 0.003)
- "human" (weight: 0.003)
- "research" (weight: 0.003)
- "may" (weight: 0.003)

Topic 2:
- "state" (weight: 0.008)
- "school" (weight: 0.007)
- "health" (weight: 0.007)
- "company" (weight: 0.007)
- "law" (weight: 0.006)
- "student" (weight: 0.006)
- "percent" (weight: 0.006)
- "care" (weight: 0.005)
- "program" (weight: 0.004)
- "federal" (weight: 0.004)

Topic 3:
- "trump" (weight: 0.034)
- "clinton" (weight: 0.014)
- "president" (weight: 0.011)
- "republican" (weight: 0.009)
- "campa

In [21]:
article_labels = []

for doc in preprocessed_documents:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({
    "Article": documents,
    "Topic": article_labels
})

print(df_result.head(10))

                                             Article  Topic
0  In the Washington of 2016, even when the polic...      3
1    Donald Trump has used Twitter  —   his prefe...      0
2    Donald Trump is unabashedly praising Russian...      3
3  Updated at 2:50 p. m. ET, Russian President Vl...      0
4  From photography, illustration and video, to d...      4
5  I did not want to join yoga class. I hated tho...      1
6  With a   who has publicly supported the debunk...      1
7  I was standing by the airport exit, debating w...      4
8  If movies were trying to be more realistic, pe...      1
9  Eighteen years ago, on New Year’s Eve, David F...      4


In [22]:
topic_names = {
    0: "Personal / Lifestyle",
    1: "US Politics / Election",
    2: "Daily Life / Social Issues",
    3: "World News / Conflict",
    4: "Health & Public Services"
}

df_result["Topic_Name"] = df_result["Topic"].map(topic_names)
print(df_result.head(10))

                                             Article  Topic  \
0  In the Washington of 2016, even when the polic...      3   
1    Donald Trump has used Twitter  —   his prefe...      0   
2    Donald Trump is unabashedly praising Russian...      3   
3  Updated at 2:50 p. m. ET, Russian President Vl...      0   
4  From photography, illustration and video, to d...      4   
5  I did not want to join yoga class. I hated tho...      1   
6  With a   who has publicly supported the debunk...      1   
7  I was standing by the airport exit, debating w...      4   
8  If movies were trying to be more realistic, pe...      1   
9  Eighteen years ago, on New Year’s Eve, David F...      4   

                 Topic_Name  
0     World News / Conflict  
1      Personal / Lifestyle  
2     World News / Conflict  
3      Personal / Lifestyle  
4  Health & Public Services  
5    US Politics / Election  
6    US Politics / Election  
7  Health & Public Services  
8    US Politics / Election  
9  Hea

In [23]:
df_result.to_csv("npr_topic_results_with_names.csv", index=False)
print("Saved successfully as npr_topic_results_with_names.csv")

Saved successfully as npr_topic_results_with_names.csv


In [24]:
discussion = """
The LDA model successfully identified meaningful topics from the NPR dataset.
Topic 1 clearly represents US politics, with strong keywords such as “Trump”,
“Clinton”, and “election”. Topic 4 focuses on healthcare and public services,
indicated by terms like “health”, “patient”, and “drug”. Topic 3 captures
world news and conflict-related issues, while Topic 0 and Topic 2 represent
more general themes such as personal lifestyle and daily social topics.
Overall, the model effectively groups related words into interpretable topics,
demonstrating the usefulness of topic modeling in analyzing large text datasets.
"""
print(discussion)


The LDA model successfully identified meaningful topics from the NPR dataset.
Topic 1 clearly represents US politics, with strong keywords such as “Trump”,
“Clinton”, and “election”. Topic 4 focuses on healthcare and public services,
indicated by terms like “health”, “patient”, and “drug”. Topic 3 captures
world news and conflict-related issues, while Topic 0 and Topic 2 represent
more general themes such as personal lifestyle and daily social topics.
Overall, the model effectively groups related words into interpretable topics,
demonstrating the usefulness of topic modeling in analyzing large text datasets.

